# Build Your First End-to-End RAG System

> **From documents to a grounded answer: building a deliberately simple RAG baseline.**

In the previous tutorial, we mapped the architecture of a Retrieval-Augmented Generation (RAG) system.

We saw that RAG is not a single model. It is a pipeline:

```text
Documents
    ↓
Ingestion
    ↓
Chunking
    ↓
Embeddings
    ↓
Indexing
    ↓
Retrieval
    ↓
Context
    ↓
LLM
    ↓
Answer
```

Now we're going to build that pipeline.

The system in this notebook is intentionally simple. The goal is not to create our final production architecture yet. The goal is to make every step visible so that we can understand what is happening.

Later in the course, we'll replace these baseline components with more robust approaches and measure whether those changes actually improve the system.

## What we're building

We'll create a small RAG system that can answer questions about a collection of text documents.

Our baseline looks like this:

```text
                 INDEXING
                    │
                    ▼
                Documents
                    │
                    ▼
                  Chunks
                    │
                    ▼
                Embeddings
                    │
                    ▼
              Vector Index
                    │
                    │
                    │
                  QUERY
                    │
                    ▼
                User Query
                    │
                    ▼
             Query Embedding
                    │
                    ▼
              Similarity Search
                    │
                    ▼
              Retrieved Chunks
                    │
                    ▼
              Context Assembly
                    │
                    ▼
                   LLM
                    │
                    ▼
                 Answer
```

By the end, we'll have implemented the core mechanics of this flow.

## A note about this baseline

This is a learning implementation, not a production implementation.

We're intentionally starting with:

- Plain text documents
- Simple fixed-size chunking
- A small embedding model
- In-memory similarity search
- A simple context construction step

We are **not** starting with every production feature at once.

That is important because a baseline gives us something we can measure and improve.

A useful engineering workflow is:

```text
Simple baseline
      ↓
Measure
      ↓
Change one component
      ↓
Measure again
      ↓
Compare
```

This approach will guide the rest of the course.

# 1. Create a Small Document Collection

To make the mechanics easy to inspect, we'll start with three small text documents.

Our example knowledge base will contain information about:

- Refunds
- Shipping
- Customer support

These are deliberately simple. Later, we'll work with real PDFs and other document formats.

### Create the dataset directory

We'll keep the raw documents under the project's `datasets/` directory.

If you're running this notebook from a different location, adjust the path accordingly.

In [1]:
from pathlib import Path

data_dir = Path("../../datasets/raw/rag-baseline")
data_dir.mkdir(parents=True, exist_ok=True)

### Create the documents

In [2]:
documents = {
    "refund_policy.txt": """
Refund requests can be submitted within 30 days of purchase.
Products must be returned in their original condition.
Refunds are normally processed within 7 business days after approval.
""",

    "shipping_policy.txt": """
Standard shipping normally takes between 3 and 5 business days.
Express shipping normally takes between 1 and 2 business days.
Customers can track their orders using the tracking number provided by email.
""",

    "support_policy.txt": """
Customer support is available Monday through Friday from 9:00 AM to 5:00 PM.
Customers can contact support through email or the customer portal.
Urgent technical problems should be reported through the priority support channel.
"""
}

Write them to disk:

In [3]:
for filename, content in documents.items():
    (data_dir / filename).write_text(content.strip())

We now have:

```text
datasets/
└── raw/
    └── rag-baseline/
        ├── refund_policy.txt
        ├── shipping_policy.txt
        └── support_policy.txt
```

# 2. Load the Documents

The first stage of our indexing pipeline is to load the source documents.

For now, these are simple `.txt` files, so Python's built-in file handling is enough.

In [5]:
documents = []

for path in data_dir.glob("*.txt"):
    text = path.read_text()

    documents.append({
        "document_id": path.stem,
        "text": text,
        "source": path.name,
    })

Let's inspect what we loaded:

In [6]:
documents

[{'document_id': 'support_policy',
  'text': 'Customer support is available Monday through Friday from 9:00 AM to 5:00 PM.\nCustomers can contact support through email or the customer portal.\nUrgent technical problems should be reported through the priority support channel.',
  'source': 'support_policy.txt'},
 {'document_id': 'shipping_policy',
  'text': 'Standard shipping normally takes between 3 and 5 business days.\nExpress shipping normally takes between 1 and 2 business days.\nCustomers can track their orders using the tracking number provided by email.',
  'source': 'shipping_policy.txt'},
 {'document_id': 'refund_policy',
  'text': 'Refund requests can be submitted within 30 days of purchase.\nProducts must be returned in their original condition.\nRefunds are normally processed within 7 business days after approval.',
  'source': 'refund_policy.txt'}]

# 3. Chunk the Documents

A document can be much larger than the amount of information we want to retrieve at once.

So we divide it into smaller pieces called **chunks**.

For this baseline, we'll use a deliberately simple fixed-size strategy.

### A simple chunker

In [7]:
def chunk_text(text, chunk_size=100):
    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks

We can test it on one document:

In [8]:
chunk_text(documents[0]["text"])

['Customer support is available Monday through Friday from 9:00 AM to 5:00 PM. Customers can contact support through email or the customer portal. Urgent technical problems should be reported through the priority support channel.']

For these tiny documents, we may only get one chunk per document. That's okay.

The important thing is understanding the transformation:

```text
Document
    ↓
Chunk 1
Chunk 2
Chunk 3
...
```

### Attach metadata to each chunk

We don't want to lose the relationship between a chunk and its original document.


In [9]:
chunks = []

for document in documents:
    for index, text in enumerate(chunk_text(document["text"])):
        chunks.append({
            "chunk_id": f'{document["document_id"]}_{index}',
            "document_id": document["document_id"],
            "source": document["source"],
            "text": text,
        })

Inspect the result:

In [10]:
chunks

[{'chunk_id': 'support_policy_0',
  'document_id': 'support_policy',
  'source': 'support_policy.txt',
  'text': 'Customer support is available Monday through Friday from 9:00 AM to 5:00 PM. Customers can contact support through email or the customer portal. Urgent technical problems should be reported through the priority support channel.'},
 {'chunk_id': 'shipping_policy_0',
  'document_id': 'shipping_policy',
  'source': 'shipping_policy.txt',
  'text': 'Standard shipping normally takes between 3 and 5 business days. Express shipping normally takes between 1 and 2 business days. Customers can track their orders using the tracking number provided by email.'},
 {'chunk_id': 'refund_policy_0',
  'document_id': 'refund_policy',
  'source': 'refund_policy.txt',
  'text': 'Refund requests can be submitted within 30 days of purchase. Products must be returned in their original condition. Refunds are normally processed within 7 business days after approval.'}]

Each chunk now carries:

```text
chunk_id
document_id
source
text
```

This is our first example of a principle that will appear throughout the course:

> **Content and metadata need to travel together through the RAG pipeline.**

# 4. Generate Embeddings

Now we need a way to represent the meaning of our chunks numerically.

We'll use a Sentence Transformers embedding model for this baseline.

Install the dependency directly from the notebook:

```python
! pip install sentence-transformers
```

In [12]:
! pip install sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 1.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 2.0 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 306.4 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 2.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 2.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 1.7 MB/s eta 0:00:0000:0100:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 1.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 979.9 kB/s eta 0:00:000:0100:08m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.0 MB/s eta 0:00:0000:0100:07m
   ━━━━━━━━━━━━━━━━━━━━━━━━━

### Load the embedding model

In [13]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Now extract the chunk text:

In [14]:
texts = [chunk["text"] for chunk in chunks]
texts

['Customer support is available Monday through Friday from 9:00 AM to 5:00 PM. Customers can contact support through email or the customer portal. Urgent technical problems should be reported through the priority support channel.',
 'Standard shipping normally takes between 3 and 5 business days. Express shipping normally takes between 1 and 2 business days. Customers can track their orders using the tracking number provided by email.',
 'Refund requests can be submitted within 30 days of purchase. Products must be returned in their original condition. Refunds are normally processed within 7 business days after approval.']

Generate the embeddings:

In [15]:
embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True
)

Inspect the shape:

In [16]:
embeddings.shape

(3, 384)

The result has the conceptual form:

```text
(number_of_chunks, embedding_dimension)
```

## What just happened?

We transformed:

```text
"Customers can request a refund within 30 days."
```

into something like:

```text
[0.021, -0.184, 0.731, ..., 0.092]
```

We don't interpret each number individually.

Instead, the vector gives us a representation that can be compared with other vectors.

This allows us to perform semantic similarity search.

# 5. Embed the User's Query

Now let's move to the query side of the RAG pipeline.

Suppose the user asks:

> **How long do I have to request a refund?**

We'll convert that query into an embedding using the same model.

In [17]:
query = "How long do I have to request a refund?"

query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True
)

We now have:

```text
Document chunks
      ↓
Chunk embeddings

User query
      ↓
Query embedding
```

The next step is to compare them.

# 6. Perform Semantic Similarity Search

Because our embeddings are normalized, we can use a dot product as a cosine-similarity calculation.

In [18]:
import numpy as np

scores = embeddings @ query_embedding

Each score represents how similar a chunk is to the query.

Let's inspect them:

In [19]:
for chunk, score in zip(chunks, scores):
    print(f"{score:.4f}  {chunk['source']}")
    print(chunk["text"])
    print()

0.6328  support_policy.txt
Customer support is available Monday through Friday from 9:00 AM to 5:00 PM. Customers can contact support through email or the customer portal. Urgent technical problems should be reported through the priority support channel.

0.6285  shipping_policy.txt
Standard shipping normally takes between 3 and 5 business days. Express shipping normally takes between 1 and 2 business days. Customers can track their orders using the tracking number provided by email.

0.8952  refund_policy.txt
Refund requests can be submitted within 30 days of purchase. Products must be returned in their original condition. Refunds are normally processed within 7 business days after approval.



### Rank the results

We want the highest-scoring chunks first.

In [20]:
ranked_indices = np.argsort(scores)[::-1]

Select the top results:

In [21]:
top_k = 3

results = [
    {
        **chunks[i],
        "score": float(scores[i])
    }
    for i in ranked_indices[:top_k]
]

Inspect the ranking:

In [22]:
for result in results:
    print(f"Score: {result['score']:.4f}")
    print(f"Source: {result['source']}")
    print(result["text"])
    print("---")

Score: 0.8952
Source: refund_policy.txt
Refund requests can be submitted within 30 days of purchase. Products must be returned in their original condition. Refunds are normally processed within 7 business days after approval.
---
Score: 0.6328
Source: support_policy.txt
Customer support is available Monday through Friday from 9:00 AM to 5:00 PM. Customers can contact support through email or the customer portal. Urgent technical problems should be reported through the priority support channel.
---
Score: 0.6285
Source: shipping_policy.txt
Standard shipping normally takes between 3 and 5 business days. Express shipping normally takes between 1 and 2 business days. Customers can track their orders using the tracking number provided by email.
---


We have now implemented the core of a simple semantic retriever.

# 7. What Is the Retriever Actually Doing?

It is worth stopping here.

A retriever is not "the vector database."

The underlying operation is:

```text
Query
  ↓
Query embedding
  ↓
Similarity calculation
  ↓
Ranking
  ↓
Relevant candidates
```

A vector database provides infrastructure for storing and searching large numbers of vectors efficiently.

For our tiny dataset, Python can calculate the similarities directly.

For a production system with millions of vectors, that approach would not be practical.

That's where systems such as Qdrant become useful.

We'll introduce Qdrant when we study vector databases.

# 8. Build the Retrieved Context

The LLM needs the retrieved information in a form it can use.

Let's combine our retrieved chunks:

In [23]:
context = "\n\n".join(
    f"Source: {result['source']}\n{result['text']}"
    for result in results
)

print(context)

Source: refund_policy.txt
Refund requests can be submitted within 30 days of purchase. Products must be returned in their original condition. Refunds are normally processed within 7 business days after approval.

Source: support_policy.txt
Customer support is available Monday through Friday from 9:00 AM to 5:00 PM. Customers can contact support through email or the customer portal. Urgent technical problems should be reported through the priority support channel.

Source: shipping_policy.txt
Standard shipping normally takes between 3 and 5 business days. Express shipping normally takes between 1 and 2 business days. Customers can track their orders using the tracking number provided by email.


We now have something like:

```text
Source: refund_policy.txt
Refund requests can be submitted within 30 days...

Source: shipping_policy.txt
Standard shipping normally takes...

Source: support_policy.txt
Customer support is available...
```

This is our **retrieved context**.

# 9. Construct the LLM Input

We can now combine the user's question with the retrieved context.

A simple prompt could look like:

In [24]:
prompt = f"""
Answer the user's question using only the supplied context.

If the context does not contain enough information to answer the
question, say that the information is not available in the context.

Context:
{context}

Question:
{query}
"""

print(prompt)


Answer the user's question using only the supplied context.

If the context does not contain enough information to answer the
question, say that the information is not available in the context.

Context:
Source: refund_policy.txt
Refund requests can be submitted within 30 days of purchase. Products must be returned in their original condition. Refunds are normally processed within 7 business days after approval.

Source: support_policy.txt
Customer support is available Monday through Friday from 9:00 AM to 5:00 PM. Customers can contact support through email or the customer portal. Urgent technical problems should be reported through the priority support channel.

Source: shipping_policy.txt
Standard shipping normally takes between 3 and 5 business days. Express shipping normally takes between 1 and 2 business days. Customers can track their orders using the tracking number provided by email.

Question:
How long do I have to request a refund?



The important transformation is:

```text
User Query
     +
Retrieved Context
     ↓
    Prompt
     ↓
     LLM
```

The LLM is now being given external information at inference time.

# 10. Generate the Answer

The final step is to send the constructed prompt to an LLM.

The exact model and inference provider are intentionally left open in this baseline.

The interface we need is simple:

```text
prompt
  ↓
LLM
  ↓
response
```

For example, a local model or an API-backed model could receive the prompt and produce:

> Refund requests can be submitted within 30 days of purchase.

The important thing is not the particular LLM yet.

The important thing is that the answer is generated **after retrieval** and uses the retrieved context.

# 11. The Complete Baseline

We have now walked through the entire RAG flow:

```text
                         INDEXING
                            │
                            ▼
                       Documents
                            │
                            ▼
                         Chunks
                            │
                            ▼
                       Embeddings
                            │
                            ▼
                     Vector Index
                            │
                            │
                            │
                          QUERY
                            │
                            ▼
                       User Query
                            │
                            ▼
                    Query Embedding
                            │
                            ▼
                    Similarity Search
                            │
                            ▼
                    Retrieved Chunks
                            │
                            ▼
                    Context Assembly
                            │
                            ▼
                           LLM
                            │
                            ▼
                         Answer
```

This is a complete, albeit minimal, RAG system.

# 12. What We Have and What We Don't Have

Our baseline already demonstrates the core RAG idea.

We have:

- Document loading
- Chunking
- Embedding
- Semantic retrieval
- Context construction
- Generation

But it is far from production-ready.

We still need to solve:

| Problem | Our baseline |
|---|---|
| Document formats | Plain text only |
| Chunking | Fixed word count |
| Vector storage | In-memory |
| Retrieval | Dense similarity only |
| Keyword search | Not implemented |
| Fusion | Not implemented |
| Reranking | Not implemented |
| Metadata filtering | Minimal |
| Provenance | Basic source name |
| Evaluation | Not implemented |
| Observability | Not implemented |
| Access control | Not implemented |

This is exactly why we built a baseline first.

# 13. Why the Baseline Matters

Imagine that we eventually build this:

```text
Robust ingestion
      ↓
Semantic chunking
      ↓
BGE-M3
      ↓
Qdrant
      ↓
Dense + BM25
      ↓
RRF
      ↓
Reranker
      ↓
Context engineering
      ↓
LLM
      ↓
Evaluation
```

If it performs better than our baseline, we can investigate **which changes produced the improvement**.

Without a baseline, "production RAG" becomes a collection of libraries and configuration choices.

With a baseline, it becomes an engineering experiment.

# 14. The Most Important Lesson

The key lesson from this notebook is not how to use Sentence Transformers.

It is the **data flow**.

A piece of information travels through the system:

```text
Source Document
      ↓
Chunk
      ↓
Embedding
      ↓
Index
      ↓
Retrieved Chunk
      ↓
Context
      ↓
LLM
      ↓
Answer
```

At every step, we should be able to ask:

> **What information entered this stage, what happened to it, and what came out?**

That mindset will be essential when we start debugging and evaluating RAG systems.

# Key Takeaways

1. A RAG system can be built as a sequence of independent stages.
2. Indexing prepares external knowledge before users query it.
3. Query-time retrieval finds information relevant to a specific question.
4. Embeddings provide one mechanism for semantic similarity search.
5. Retrieved chunks become context for the LLM.
6. A vector database is infrastructure for efficient vector storage and search; it is not the definition of RAG.
7. Metadata should remain attached to chunks so that provenance can be preserved.
8. A simple baseline gives us something to measure before adding complexity.

Our baseline is intentionally small.

The next step is to understand the **indexing pipeline itself in much greater depth**: what happens to documents before they become searchable, what exactly should be stored, and how indexing decisions affect retrieval later.